# 02 — Weekly Data Update

This notebook updates the in season data used by the weekly projection model.

The original preseason notebooks and their outputs remain unchanged. This notebook only reads from the frozen preseason model and collects new regular season information for the weekly projection layer.

For a selected target week, the notebook:

- Downloads the current 2026 NFL schedule/results
- Keeps only completed games from weeks before the target week
- Saves the upcoming target-week schedule
- Preserves the original preseason projection for the target week as a comparison baseline
- Saves all weekly data into separate in-season files

No in season feature engineering or team strength updating is performed here. Those steps belong in later weekly notebooks.


In [1]:
from pathlib import Path

import pandas as pd
import nflreadpy as nfl


## Project Paths

This notebook lives inside:

`weekly_projections/notebooks/`

so the project root is two directories above the notebook.

All in season intermediate data is stored separately under:

`data/processed/weekly/`

The original preseason outputs are only read, never overwritten.


In [2]:
PROJECT_ROOT = Path("../..")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SCHEDULE_DIR = DATA_DIR / "schedules"

WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"
WEEKLY_OUTPUT_DIR = (
    PROJECT_ROOT
    / "weekly_projections"
    / "outputs"
)

WEEKLY_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

WEEKLY_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT.resolve())
print("Weekly data:", WEEKLY_DATA_DIR.resolve())


Project root: C:\Users\efriedman\Desktop\NFL-Season-Projections
Weekly data: C:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\weekly


## Weekly Settings

`TARGET_WEEK` is the week we are preparing to project.

For Week 2:

- Week 1 completed games may be used
- Week 2 results may not be used
- The preseason model remains the prior/baseline

Next week, this value can simply be changed from `2` to `3`.


In [3]:
SEASON = 2026
TARGET_WEEK = 3

PRIOR_WEEK = TARGET_WEEK - 1

print(f"Season: {SEASON}")
print(f"Target week: {TARGET_WEEK}")
print(f"Completed data allowed through Week {PRIOR_WEEK}")


Season: 2026
Target week: 3
Completed data allowed through Week 2


# Current 2026 Schedule and Results

The schedule is pulled fresh from nflverse through `nflreadpy`.

A game counts as completed only when both the home and away scores are available.

This allows the notebook to be rerun at any point. If a game from the prior week has not finished yet, it will automatically be excluded until a final score is available.


In [4]:
schedule_2026 = nfl.load_schedules(
    seasons=[SEASON]
).to_pandas()

schedule_2026 = schedule_2026[
    schedule_2026["game_type"] == "REG"
].copy()

schedule_2026 = schedule_2026.sort_values(
    ["week", "gameday", "game_id"]
).reset_index(drop=True)

print("2026 regular-season games:", len(schedule_2026))
print("Columns:", len(schedule_2026.columns))

display(
    schedule_2026[
        [
            "week",
            "gameday",
            "away_team",
            "home_team",
            "away_score",
            "home_score"
        ]
    ].head(20)
)


2026 regular-season games: 272
Columns: 46


,week,gameday,away_team,home_team,away_score,home_score
0,1,2026-09-09,NE,SEA,10.0,13.0
1,1,2026-09-10,SF,LA,27.0,7.0
2,1,2026-09-13,ARI,LAC,26.0,14.0
3,1,2026-09-13,ATL,PIT,13.0,20.0
4,1,2026-09-13,BAL,IND,41.0,23.0
5,1,2026-09-13,BUF,HOU,36.0,31.0
6,1,2026-09-13,CHI,CAR,59.0,37.0
7,1,2026-09-13,CLE,JAX,10.0,34.0
8,1,2026-09-13,DAL,NYG,20.0,28.0
9,1,2026-09-13,GB,MIN,22.0,39.0


## Save a Current Schedule Snapshot

This is a weekly layer snapshot only. It does not replace or modify the historical schedule files created by the preseason pipeline.


In [5]:
current_schedule_path = (
    WEEKLY_DATA_DIR
    / f"{SEASON}_schedule_current.parquet"
)

schedule_2026.to_parquet(
    current_schedule_path,
    index=False
)

print("Saved:", current_schedule_path)


Saved: ..\..\data\processed\weekly\2026_schedule_current.parquet


# Completed Games Available to the Weekly Model

Only games from weeks strictly before `TARGET_WEEK` are eligible.

This is the core leakage prevention rule for the weekly model.


In [6]:
completed_games = schedule_2026[
    (schedule_2026["week"] < TARGET_WEEK)
    & (schedule_2026["home_score"].notna())
    & (schedule_2026["away_score"].notna())
].copy()

completed_games = completed_games.sort_values(
    ["week", "gameday", "game_id"]
).reset_index(drop=True)

print(
    f"Completed games available before Week {TARGET_WEEK}:",
    len(completed_games)
)

display(
    completed_games[
        [
            "week",
            "gameday",
            "away_team",
            "away_score",
            "home_team",
            "home_score"
        ]
    ]
)


Completed games available before Week 3: 32


,week,gameday,away_team,away_score,home_team,home_score
0,1,2026-09-09,NE,10.0,SEA,13.0
1,1,2026-09-10,SF,27.0,LA,7.0
2,1,2026-09-13,ARI,26.0,LAC,14.0
3,1,2026-09-13,ATL,13.0,PIT,20.0
4,1,2026-09-13,BAL,41.0,IND,23.0
5,1,2026-09-13,BUF,36.0,HOU,31.0
6,1,2026-09-13,CHI,59.0,CAR,37.0
7,1,2026-09-13,CLE,10.0,JAX,34.0
8,1,2026-09-13,DAL,20.0,NYG,28.0
9,1,2026-09-13,GB,22.0,MIN,39.0


## Completion Check

Before generating the final Week 2 model, Week 1 should contain all 16 completed games.

If this notebook is run before the final Week 1 game has finished, the warning below is expected. Simply rerun the notebook after that game becomes final.


In [7]:
prior_week_schedule = schedule_2026[
    schedule_2026["week"] == PRIOR_WEEK
].copy()

prior_week_completed = prior_week_schedule[
    prior_week_schedule["home_score"].notna()
    & prior_week_schedule["away_score"].notna()
].copy()

prior_week_total = len(prior_week_schedule)
prior_week_completed_count = len(prior_week_completed)

print(
    f"Week {PRIOR_WEEK}: "
    f"{prior_week_completed_count}/{prior_week_total} games completed"
)

if prior_week_completed_count < prior_week_total:
    unfinished = prior_week_schedule[
        prior_week_schedule["home_score"].isna()
        | prior_week_schedule["away_score"].isna()
    ]

    print()
    print(
        "WARNING: The prior week is not fully complete yet."
    )
    print(
        "Do not lock the next week's updated ratings until "
        "all prior week games are final."
    )

    display(
        unfinished[
            [
                "gameday",
                "away_team",
                "home_team"
            ]
        ]
    )
else:
    print(
        f"Week {PRIOR_WEEK} is complete. "
        f"Week {TARGET_WEEK} data can be locked."
    )


Week 2: 16/16 games completed
Week 2 is complete. Week 3 data can be locked.


# Target Week Schedule

This contains the games we are preparing to project.

Sportsbook fields are retained if available in the nflverse schedule, but they will be used only for model-versus-market comparison and not as inputs to team strength.


In [8]:
target_week_schedule = schedule_2026[
    schedule_2026["week"] == TARGET_WEEK
].copy()

target_week_schedule = target_week_schedule.sort_values(
    ["gameday", "game_id"]
).reset_index(drop=True)

print(
    f"Week {TARGET_WEEK} games:",
    len(target_week_schedule)
)

display_columns = [
    "gameday",
    "away_team",
    "home_team"
]

for optional_column in [
    "away_rest",
    "home_rest",
    "spread_line",
    "total_line"
]:
    if optional_column in target_week_schedule.columns:
        display_columns.append(optional_column)

display(
    target_week_schedule[
        display_columns
    ]
)


Week 3 games: 16


,gameday,away_team,home_team,away_rest,home_rest,spread_line,total_line
0,2026-09-24,ATL,GB,4,4,6.0,44.5
1,2026-09-27,ARI,SF,7,7,8.5,47.5
2,2026-09-27,BAL,DAL,7,7,-3.0,52.5
3,2026-09-27,CAR,CLE,7,7,-2.5,42.5
4,2026-09-27,CIN,PIT,7,7,-3.5,42.5
5,2026-09-27,HOU,IND,7,7,-2.5,43.5
6,2026-09-27,KC,MIA,7,7,-11.5,46.5
7,2026-09-27,LAC,BUF,7,10,7.0,50.5
8,2026-09-27,LA,DEN,6,7,-2.5,45.5
9,2026-09-27,LV,NO,7,7,3.0,43.5


# Frozen Preseason Baseline for the Target Week

The original `2026_game_predictions.parquet` file is the output of the preseason model.

We extract the selected week's preseason predictions here so that future weekly projections can always be compared with what the model believed before the season began.

The original file is not changed.


In [9]:
preseason_predictions = pd.read_parquet(
    PROCESSED_DIR
    / "2026_game_predictions.parquet"
)

target_week_preseason = preseason_predictions[
    preseason_predictions["week"] == TARGET_WEEK
].copy()

target_week_preseason = target_week_preseason.sort_values(
    ["gameday", "game_id"]
).reset_index(drop=True)

print(
    f"Frozen preseason Week {TARGET_WEEK} predictions:",
    len(target_week_preseason)
)

display(
    target_week_preseason[
        [
            "gameday",
            "away_team",
            "home_team",
            "away_team_strength",
            "home_team_strength",
            "home_field_adjustment",
            "rest_adjustment",
            "expected_home_margin",
            "predicted_winner",
            "predicted_win_probability"
        ]
    ]
)


Frozen preseason Week 3 predictions: 16


,gameday,away_team,home_team,away_team_strength,home_team_strength,home_field_adjustment,rest_adjustment,expected_home_margin,predicted_winner,predicted_win_probability
0,2026-09-24,ATL,GB,-1.331272,2.536317,1.763528,0.000000,5.631117,GB,0.654145
1,2026-09-27,ARI,SF,-3.678016,2.366788,1.763528,0.000000,7.808331,SF,0.708790
2,2026-09-27,BAL,DAL,3.074111,-0.386396,0.000000,0.000000,-3.460507,BAL,0.596262
3,2026-09-27,CAR,CLE,-4.533242,-4.150395,1.763528,0.000000,2.146375,CLE,0.560069
4,2026-09-27,CIN,PIT,-0.739434,0.543902,1.763528,0.000000,3.046864,PIT,0.584943
5,2026-09-27,HOU,IND,2.810133,0.532226,1.763528,0.000000,-0.514379,HOU,0.514447
6,2026-09-27,KC,MIA,1.302244,-2.443254,1.763528,0.000000,-1.981971,KC,0.555499
7,2026-09-27,LAC,BUF,0.910681,4.816514,1.763528,0.542984,6.212344,BUF,0.669113
8,2026-09-27,LA,DEN,4.762452,3.569507,1.763528,0.180995,0.751578,DEN,0.521104
9,2026-09-27,LV,NO,-5.693342,-1.556053,1.763528,0.000000,5.900817,NO,0.661122


# Data Quality Checks

These checks make sure the weekly layer has a clean foundation before feature engineering begins.


In [10]:
expected_teams = set(
    pd.concat(
        [
            schedule_2026["home_team"],
            schedule_2026["away_team"]
        ]
    ).dropna().unique()
)

completed_teams = set(
    pd.concat(
        [
            completed_games["home_team"],
            completed_games["away_team"]
        ]
    ).dropna().unique()
)

print("Teams in 2026 schedule:", len(expected_teams))
print("Teams represented in completed games:", len(completed_teams))

missing_from_completed = sorted(
    expected_teams - completed_teams
)

if missing_from_completed:
    print(
        "Teams without a completed game yet:",
        missing_from_completed
    )
else:
    print(
        "Every team has at least one completed game."
    )

print()
print(
    "Duplicate completed game IDs:",
    completed_games["game_id"].duplicated().sum()
)

print(
    "Missing completed game scores:",
    completed_games[
        ["home_score", "away_score"]
    ].isna().sum().sum()
)


Teams in 2026 schedule: 32
Teams represented in completed games: 32
Every team has at least one completed game.

Duplicate completed game IDs: 0
Missing completed game scores: 0


# Save Weekly Data Inputs

The saved files become the inputs to the next notebooks:

- `03_Inseason_Feature_Engineering.ipynb`
- `04_Weekly_Team_Strength.ipynb`
- `05_Weekly_Game_Predictions.ipynb`

The filenames explicitly identify the target week so previous weeks remain reproducible.


In [11]:
completed_games_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_completed_games.parquet"
)

target_schedule_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_schedule.parquet"
)

preseason_baseline_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_preseason_baseline.parquet"
)

completed_games.to_parquet(
    completed_games_path,
    index=False
)

target_week_schedule.to_parquet(
    target_schedule_path,
    index=False
)

target_week_preseason.to_parquet(
    preseason_baseline_path,
    index=False
)

print("Saved:")
print(completed_games_path)
print(target_schedule_path)
print(preseason_baseline_path)


Saved:
..\..\data\processed\weekly\week_03_completed_games.parquet
..\..\data\processed\weekly\week_03_schedule.parquet
..\..\data\processed\weekly\week_03_preseason_baseline.parquet


# Weekly Data Update Summary

This notebook creates a clean separation between the preseason model and the in season update process.

For the selected target week:

1. The current NFL schedule and results are downloaded.
2. Only completed games from earlier weeks are made available to the model.
3. The upcoming schedule is saved separately.
4. The original preseason projection for that week is preserved as a frozen comparison baseline.
5. No original preseason notebook or output is overwritten.

The next notebook will use the completed game file to create in season performance features while keeping all preseason ratings intact.
